In [3]:
from __future__ import annotations



from pathlib import Path

import json

import shutil

import subprocess



import numpy as np

from PIL import Image, ImageDraw, ImageFilter, ImageFont





OUTPUT_FORMAT = "webm"  # "webm" | "gif" | "mp4"

FPS = 24

LOOP_FRAMES = 180

TOTAL_FRAMES = LOOP_FRAMES



ANIMATIONS_DIR = Path("media-site/animations")

ANIMATION_NAME = "telemetry_celestial_sphere_3d"



STYLE_PATH = Path("style/SA_styles.json")

CANVAS_SIZE = (900, 900)



BG_DARK = (2, 7, 13)

CYAN = (90, 240, 255)

CYAN2 = (180, 255, 255)

GREEN = (90, 255, 170)

YELLOW = (255, 220, 90)

RED = (255, 80, 110)

WHITE = (245, 250, 255)





def load_style(path: Path) -> dict:

    if not path.exists():

        return {}

    try:

        return json.loads(path.read_text(encoding="utf-8"))

    except Exception:

        return {}





def hex_to_rgb(value: str | None, fallback: tuple[int, int, int]) -> tuple[int, int, int]:

    if not value:

        return fallback

    value = value.strip().lstrip("#")

    if len(value) != 6:

        return fallback

    try:

        return tuple(int(value[i:i + 2], 16) for i in (0, 2, 4))

    except Exception:

        return fallback





STYLE = load_style(STYLE_PATH)



SA_CYAN = hex_to_rgb(STYLE.get("cyan"), CYAN)

SA_CYAN2 = hex_to_rgb(STYLE.get("cyan2"), CYAN2)

SA_GREEN = hex_to_rgb(STYLE.get("green"), GREEN)

SA_YELLOW = hex_to_rgb(STYLE.get("yellow"), YELLOW)

SA_BG = hex_to_rgb(STYLE.get("background"), BG_DARK)





def load_font(size: int):

    candidates = [

        "/System/Library/Fonts/Menlo.ttc",

        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",

        "/Library/Fonts/Arial.ttf",

        "DejaVuSansMono.ttf",

    ]



    for path in candidates:

        try:

            return ImageFont.truetype(path, size)

        except Exception:

            pass



    return ImageFont.load_default()





def normalize_output_format(fmt: str) -> str:

    fmt = fmt.lower().strip()

    if fmt not in {"webm", "gif", "mp4"}:

        raise ValueError("OUTPUT_FORMAT must be webm, gif or mp4")

    return fmt





def loop_phase(i: int, cycles: int = 1) -> float:

    """

    Seamless loop phase.



    Last generated frame is NOT equal to first frame.

    Next playback frame after i=TOTAL_FRAMES-1 is i=0,

    so integer cycles close cleanly.

    """

    return 2 * np.pi * cycles * i / TOTAL_FRAMES





def make_canvas(output_format: str) -> Image.Image:

    if normalize_output_format(output_format) == "webm":

        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))

    return Image.new("RGBA", CANVAS_SIZE, (*SA_BG, 255))





def flatten_to_black(frame: Image.Image) -> Image.Image:

    frame = frame.convert("RGBA")

    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))

    bg.alpha_composite(frame)

    return bg.convert("RGB")





def rot_x(v: np.ndarray, a: float) -> np.ndarray:

    c, s = np.cos(a), np.sin(a)

    return np.array([

        v[0],

        c * v[1] - s * v[2],

        s * v[1] + c * v[2],

    ])





def rot_y(v: np.ndarray, a: float) -> np.ndarray:

    c, s = np.cos(a), np.sin(a)

    return np.array([

        c * v[0] + s * v[2],

        v[1],

        -s * v[0] + c * v[2],

    ])





def rot_z(v: np.ndarray, a: float) -> np.ndarray:

    c, s = np.cos(a), np.sin(a)

    return np.array([

        c * v[0] - s * v[1],

        s * v[0] + c * v[1],

        v[2],

    ])





def sphere_point(lon: float, lat: float) -> np.ndarray:

    return np.array([

        np.cos(lat) * np.cos(lon),

        np.cos(lat) * np.sin(lon),

        np.sin(lat),

    ])





class Camera:

    def __init__(self, w: int, h: int):

        self.w = w

        self.h = h

        self.cx = w * 0.5

        self.cy = h * 0.52

        self.scale = min(w, h) * 0.34



    def project(self, p: np.ndarray) -> tuple[float, float, float]:

        p = rot_x(p, np.deg2rad(-18))

        p = rot_z(p, np.deg2rad(-18))



        distance = 4.2

        z = p[2] + distance

        k = distance / z



        x = self.cx + p[0] * self.scale * k

        y = self.cy - p[1] * self.scale * k



        return x, y, p[2]





def draw_polyline(

    img: Image.Image,

    cam: Camera,

    points: list[np.ndarray],

    color: tuple[int, int, int],

    alpha: int,

    width: int = 1,

    hide_back: bool = True,

):

    d = ImageDraw.Draw(img)

    segment = []



    for p in points:

        x, y, depth = cam.project(p)



        if (not hide_back) or depth >= -0.18:

            segment.append((x, y))

        else:

            if len(segment) > 1:

                d.line(segment, fill=(*color, alpha), width=width)

            segment = []



    if len(segment) > 1:

        d.line(segment, fill=(*color, alpha), width=width)





def draw_circle_3d(

    img: Image.Image,

    cam: Camera,

    transform,

    color: tuple[int, int, int],

    alpha: int,

    width: int = 1,

    hide_back: bool = True,

):

    pts = []

    for t in np.linspace(0, 2 * np.pi, 360):

        p = np.array([np.cos(t), np.sin(t), 0.0])

        pts.append(transform(p))

    draw_polyline(img, cam, pts, color, alpha, width, hide_back)





def draw_latitude(

    img: Image.Image,

    cam: Camera,

    lat: float,

    transform,

    color: tuple[int, int, int],

    alpha: int,

    width: int = 1,

):

    pts = []

    for lon in np.linspace(0, 2 * np.pi, 360):

        pts.append(transform(sphere_point(lon, lat)))

    draw_polyline(img, cam, pts, color, alpha, width)





def draw_longitude(

    img: Image.Image,

    cam: Camera,

    lon: float,

    transform,

    color: tuple[int, int, int],

    alpha: int,

    width: int = 1,

):

    pts = []

    for lat in np.linspace(-np.pi / 2, np.pi / 2, 240):

        pts.append(transform(sphere_point(lon, lat)))

    draw_polyline(img, cam, pts, color, alpha, width)





def draw_axis(

    img: Image.Image,

    cam: Camera,

    p0: np.ndarray,

    p1: np.ndarray,

    color: tuple[int, int, int],

    alpha: int,

    width: int = 2,

):

    d = ImageDraw.Draw(img)

    x0, y0, _ = cam.project(p0)

    x1, y1, _ = cam.project(p1)

    d.line([x0, y0, x1, y1], fill=(*color, alpha), width=width)

    d.ellipse([x1 - 5, y1 - 5, x1 + 5, y1 + 5], fill=(*color, alpha))





def draw_text_label(

    img: Image.Image,

    cam: Camera,

    p: np.ndarray,

    text: str,

    font,

    color: tuple[int, int, int],

    alpha: int = 220,

    dx: int = 8,

    dy: int = -8,

):

    d = ImageDraw.Draw(img)

    x, y, _ = cam.project(p)

    d.text((x + dx, y + dy), text, font=font, fill=(*color, alpha))





def draw_horizon_disc(img: Image.Image, cam: Camera):

    layer = Image.new("RGBA", img.size, (0, 0, 0, 0))

    d = ImageDraw.Draw(layer)



    pts = []

    for t in np.linspace(0, 2 * np.pi, 360):

        p = np.array([np.cos(t), np.sin(t), 0.0])

        x, y, _ = cam.project(p)

        pts.append((x, y))



    # outline only — avoid solid cyan disc fill in webm

    d.line(pts + [pts[0]], fill=(*SA_CYAN2, 170), width=2)



    img.alpha_composite(layer)





def draw_globe_shell(img: Image.Image, cam: Camera):

    layer = Image.new("RGBA", img.size, (0, 0, 0, 0))

    d = ImageDraw.Draw(layer)



    cx, cy = cam.cx, cam.cy

    rx = cam.scale * 1.02

    ry = cam.scale * 0.82



    d.ellipse(

        [cx - rx, cy - ry, cx + rx, cy + ry],

        outline=(*SA_CYAN2, 80),

        width=2,

    )



    glow = layer.filter(ImageFilter.GaussianBlur(6))

    img.alpha_composite(glow)

    img.alpha_composite(layer)





def draw_hud_frame(img: Image.Image, font):

    d = ImageDraw.Draw(img)

    w, h = img.size



    pad = 34

    cut = 50



    pts = [

        (pad + cut, pad),

        (w - pad, pad),

        (w - pad, h - pad - cut),

        (w - pad - cut, h - pad),

        (pad, h - pad),

        (pad, pad + cut),

    ]



    d.line(pts + [pts[0]], fill=(*SA_CYAN, 130), width=2)



    d.rectangle(

        [pad + 22, pad + 22, w - pad - 22, h - pad - 22],

        outline=(*SA_CYAN2, 42),

        width=1,

    )



    d.text(

        (pad + 22, pad + 14),

        "TELEMETRY // CELESTIAL SPHERE 3D",

        font=font,

        fill=(*SA_CYAN2, 210),

    )





def draw_precession_cone(img: Image.Image, cam: Camera, phase: float, font):

    cone_tilt = np.deg2rad(23.5)

    cone_radius = np.sin(cone_tilt)

    cone_z = np.cos(cone_tilt)



    ring = []

    for a in np.linspace(0, 2 * np.pi, 360):

        p = np.array([

            cone_radius * np.cos(a),

            cone_radius * np.sin(a),

            cone_z,

        ])

        p = rot_x(p, np.deg2rad(-12))

        ring.append(p)



    draw_polyline(img, cam, ring, SA_YELLOW, 120, width=1, hide_back=False)



    pole = np.array([

        cone_radius * np.cos(phase),

        cone_radius * np.sin(phase),

        cone_z,

    ])

    pole = rot_x(pole, np.deg2rad(-12))



    draw_axis(img, cam, np.array([0.0, 0.0, 0.0]), pole * 1.18, SA_YELLOW, 230, width=3)

    draw_text_label(img, cam, pole * 1.22, "POLE / PRECESSION", font, SA_YELLOW)





def draw_grid_systems(img: Image.Image, cam: Camera, phase: float):

    # Local horizontal system: cyan

    def local_transform(p: np.ndarray) -> np.ndarray:

        return rot_z(p, phase)



    for lat in np.deg2rad(np.arange(-60, 75, 15)):

        draw_latitude(img, cam, lat, local_transform, SA_CYAN, 34)



    for lon in np.deg2rad(np.arange(0, 360, 15)):

        draw_longitude(img, cam, lon, local_transform, SA_CYAN, 28)



    draw_circle_3d(img, cam, local_transform, SA_CYAN2, 190, width=2)



    # Equatorial/parallactic tilted system: green

    tilt = np.deg2rad(23.5)



    def tilted_transform(p: np.ndarray) -> np.ndarray:

        p = rot_x(p, tilt)

        return rot_z(p, -phase + 0.55)



    for lat in np.deg2rad([-60, -30, 0, 30, 60]):

        draw_latitude(img, cam, lat, tilted_transform, SA_GREEN, 60)



    for lon in np.deg2rad(np.arange(0, 360, 30)):

        draw_longitude(img, cam, lon, tilted_transform, SA_GREEN, 44)



    draw_circle_3d(img, cam, tilted_transform, SA_GREEN, 200, width=2)



    # Meridian

    def meridian_transform(p: np.ndarray) -> np.ndarray:

        return rot_y(p, np.pi / 2)



    draw_circle_3d(img, cam, meridian_transform, RED, 190, width=2, hide_back=False)



    # Target point: closed-loop motion

    target = sphere_point(

        phase,

        np.deg2rad(28 + 8 * np.sin(phase)),

    )

    target = tilted_transform(target)



    d = ImageDraw.Draw(img)

    x, y, _ = cam.project(target)

    d.ellipse(

        [x - 8, y - 8, x + 8, y + 8],

        fill=(*WHITE, 230),

        outline=(*SA_GREEN, 240),

        width=1,

    )



    return target





def draw_footer_labels(img: Image.Image, font):

    d = ImageDraw.Draw(img)

    w, h = img.size



    d.text((72, h - 126), "CYAN  : AZIMUTH / ALTITUDE GRID", font=font, fill=(*SA_CYAN2, 185))

    d.text((72, h - 100), "GREEN : PARALLACTIC / EQUATORIAL GRID", font=font, fill=(*SA_GREEN, 185))

    d.text((72, h - 74), "RED   : MERIDIAN", font=font, fill=(*RED, 185))

    d.text((72, h - 48), "YELLOW: PRECESSION CONE", font=font, fill=(*SA_YELLOW, 185))





def draw_frame(i: int, output_format: str) -> Image.Image:

    frame = make_canvas(output_format)



    font_big = load_font(22)

    font_small = load_font(15)



    cam = Camera(*CANVAS_SIZE)



    phase = loop_phase(i, cycles=1)



    draw_hud_frame(frame, font_big)



    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))



    draw_globe_shell(scene, cam)

    draw_horizon_disc(scene, cam)

    draw_grid_systems(scene, cam, phase)

    draw_precession_cone(scene, cam, phase, font_small)



    # cardinal points remain static

    draw_text_label(scene, cam, np.array([0, 1.12, 0]), "NORTH", font_small, SA_CYAN2)

    draw_text_label(scene, cam, np.array([0, -1.12, 0]), "SOUTH", font_small, SA_CYAN2)

    draw_text_label(scene, cam, np.array([1.12, 0, 0]), "EAST", font_small, SA_CYAN2)

    draw_text_label(scene, cam, np.array([-1.12, 0, 0]), "WEST", font_small, SA_CYAN2)



    glow = scene.filter(ImageFilter.GaussianBlur(4))



    frame.alpha_composite(glow)

    frame.alpha_composite(scene)



    draw_footer_labels(frame, font_small)



    return frame





def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:

    if shutil.which("ffmpeg") is None:

        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")



    out_path.parent.mkdir(parents=True, exist_ok=True)

    frame_dir = out_path.parent / "_frames"

    frame_dir.mkdir(parents=True, exist_ok=True)



    for idx, frame in enumerate(frames):

        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")



    subprocess.run(

        [

            "ffmpeg", "-y",

            "-framerate", str(fps),

            "-i", str(frame_dir / "frame_%04d.png"),

            "-c:v", "libvpx-vp9",

            "-b:v", "0",

            "-crf", "34",

            "-pix_fmt", "yuva420p",

            "-auto-alt-ref", "0",

            "-row-mt", "1",

            str(out_path),

        ],

        check=True,

    )



    if not keep_frames:

        shutil.rmtree(frame_dir, ignore_errors=True)



    return out_path





def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:

    if shutil.which("ffmpeg") is None:

        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")



    out_path.parent.mkdir(parents=True, exist_ok=True)

    frame_dir = out_path.parent / "_frames"

    frame_dir.mkdir(parents=True, exist_ok=True)



    for idx, frame in enumerate(frames):

        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")



    subprocess.run(

        [

            "ffmpeg", "-y",

            "-framerate", str(fps),

            "-i", str(frame_dir / "frame_%04d.png"),

            "-c:v", "libx264",

            "-crf", "22",

            "-pix_fmt", "yuv420p",

            "-movflags", "+faststart",

            str(out_path),

        ],

        check=True,

    )



    if not keep_frames:

        shutil.rmtree(frame_dir, ignore_errors=True)



    return out_path





def export_gif(frames: list[Image.Image], out_path: Path, fps: int) -> Path:

    out_path.parent.mkdir(parents=True, exist_ok=True)



    gif_frames = [

        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)

        for frame in frames

    ]



    gif_frames[0].save(

        out_path,

        save_all=True,

        append_images=gif_frames[1:],

        duration=int(1000 / fps),

        loop=0,

        disposal=2,

    )



    return out_path





def export_animation(frames: list[Image.Image], output_format: str) -> Path:

    output_format = normalize_output_format(output_format)



    folder = ANIMATIONS_DIR / ANIMATION_NAME

    folder.mkdir(parents=True, exist_ok=True)



    out_path = folder / f"{ANIMATION_NAME}.{output_format}"



    if output_format == "webm":

        return export_webm(frames, out_path, FPS)



    if output_format == "mp4":

        return export_mp4(frames, out_path, FPS)



    return export_gif(frames, out_path, FPS)





def main():

    output_format = normalize_output_format(OUTPUT_FORMAT)



    print("[START] 3D celestial sphere telemetry")

    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")

    print(f"[CONFIG] FPS = {FPS}")

    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")

    print(f"[CONFIG] STYLE_PATH = {STYLE_PATH}")

    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")



    frames = []



    for i in range(TOTAL_FRAMES):

        if i % 24 == 0:

            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")

        frames.append(draw_frame(i, output_format))



    saved_path = export_animation(frames, output_format)



    print()

    print(f"[CREATED] {ANIMATION_NAME}")

    print(f"          {saved_path.resolve()}")

    print()

    print(

        f"Created 1 animation(s) as '{output_format}' "

        f"in '{ANIMATIONS_DIR.resolve()}'"

    )





main()


[START] 3D celestial sphere telemetry
[CONFIG] OUTPUT_FORMAT = gif
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 180
[CONFIG] STYLE_PATH = style/SA_styles.json
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/180
[GENERATE] frame 24/180
[GENERATE] frame 48/180
[GENERATE] frame 72/180
[GENERATE] frame 96/180
[GENERATE] frame 120/180
[GENERATE] frame 144/180
[GENERATE] frame 168/180

[CREATED] telemetry_celestial_sphere_3d
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/telemetry_celestial_sphere_3d/telemetry_celestial_sphere_3d.gif

Created 1 animation(s) as 'gif' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'
